In [64]:
import os 
import joblib
import pickle


import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler_directory = '/data/Hydra_Work/Sep_2024_Data/scalers'
if not os.path.exists(scaler_directory):
    os.makedirs(scaler_directory)


era5_scaler_filename = os.path.join(scaler_directory, 'era5_scaler.save')
Flat_Era5 = pd.concat(era5.values(), ignore_index=True)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(Flat_Era5)
joblib.dump(scaler, era5_scaler_filename)



In [95]:
scaler_directory = '/data/Hydra_Work/Sep_2024_Data/Sep_05_scalers'
if not os.path.exists(scaler_directory):
    os.makedirs(scaler_directory)

specific_daily_flow = daily_flow.copy()
for basin, df in specific_daily_flow.items():

    df['00060'] = pd.to_numeric(df['00060'], errors='coerce')

    # Drop rows with NaN values in '00060'
    df = df.dropna(subset=['00060'])
    specific_daily_flow[basin] = specific_daily_flow[basin]['00060']/static_variables[static_variables.index == basin]['drainage_area'][0]


flat_daily_flow = pd.concat(specific_daily_flow.values())
log_flow = np.log(flat_daily_flow.values.squeeze())
scaler = StandardScaler()
scaled_data = scaler.fit_transform(log_flow.reshape(-1, 1))
daily_flow_scaler_filename = os.path.join(scaler_directory, 'daily_flow_scaler.save')

joblib.dump(scaler, daily_flow_scaler_filename)

hungry_horse_reservoir_inflow
snake_r_nr_heise
libby_reservoir_inflow
boise_r_nr_boise
green_r_bl_howard_a_hanson_dam
weber_r_nr_oakley
san_joaquin_river_millerton_reservoir
merced_river_yosemite_at_pohono_bridge
colville_r_at_kettle_falls
stehekin_r_at_stehekin
detroit_lake_inflow
owyhee_r_bl_owyhee_dam
pueblo_reservoir_inflow
sweetwater_r_nr_alcova
missouri_r_at_toston
animas_r_at_durango
yampa_r_nr_maybell
taylor_park_reservoir_inflow
dillon_reservoir_inflow
virgin_r_at_virtin
boysen_reservoir_inflow
pecos_r_nr_pecos


/tmp/ipykernel_1435031/2711032169.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  specific_daily_flow[basin] = specific_daily_flow[basin]['00060']/static_variables[static_variables.index == basin]['drainage_area'][0]
/tmp/ipykernel_1435031/2711032169.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  specific_daily_flow[basin] = specific_daily_flow[basin]['00060']/static_variables[static_variables.index == basin]['drainage_area'][0]
/tmp/ipykernel_1435031/2711032169.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as lab

['/data/Hydra_Work/Sep_2024_Data/Sep_05_scalers/daily_flow_scaler.save']

In [4]:
import pandas as pd

static_variables_filename = '/data/Hydra_Work/Sep_2024_Data/static_variables_original.csv'
static_variables = pd.read_csv(static_variables_filename, index_col=0)
site_id = pd.read_csv('/data/Hydra_Work/Sep_2024_Data/site_id_metadata.csv', index_col=0)
site_id = site_id.drop('skagit_ross_reservoir')
site_id = site_id[['usgs_id']]
site_id['usgs_id'] = site_id['usgs_id'].apply(lambda x: f"{int(x):08}")


daily_flow = pd.read_pickle('/data/Hydra_Work/Sep_2024_Data/USGS_dataframes.pkl') # Has Swwetwater but not Fonetenelle
daily_flow = {site_id[site_id['usgs_id'] == key].index[0]: value for key, value in daily_flow.items()}

for key, df in daily_flow.items():
    daily_flow[key].index = pd.to_datetime(df['datetime'])



In [11]:
daily_flow[ 'green_r_bl_howard_a_hanson_dam']['00060']

'999'

{'12362500':      agency_cd   site_no   datetime 00060 cd
 1         USGS  12362500 2000-08-30  2600  A
 2         USGS  12362500 2000-08-31  2200  A
 3         USGS  12362500 2000-09-01  2500  A
 4         USGS  12362500 2000-09-02  2510  A
 5         USGS  12362500 2000-09-03  2350  A
 ...        ...       ...        ...   ... ..
 8762      USGS  12362500 2024-08-25  2730  P
 8763      USGS  12362500 2024-08-26  2730  P
 8764      USGS  12362500 2024-08-27  2720  P
 8765      USGS  12362500 2024-08-28  2720  P
 8766      USGS  12362500 2024-08-29  2700  P
 
 [8766 rows x 5 columns],
 '13037500':      agency_cd   site_no   datetime 00060 cd
 1         USGS  13037500 2000-08-30  9540  A
 2         USGS  13037500 2000-08-31  9540  A
 3         USGS  13037500 2000-09-01  9250  A
 4         USGS  13037500 2000-09-02  8820  A
 5         USGS  13037500 2000-09-03  8630  A
 ...        ...       ...        ...   ... ..
 8762      USGS  13037500 2024-08-25  8030  P
 8763      USGS  13037500 20

In [96]:
daily_flow_scaler = joblib.load(daily_flow_scaler_filename)

specific_daily_flow = daily_flow.copy()
for basin, df in specific_daily_flow.items():
    df['00060'] = df['00060'].astype(float)
    df = df[['00060']]

    drainage_area = static_variables.loc[basin, 'drainage_area']
    # Compute the log-transformed values
    log_transformed_values = np.log(df['00060'] / drainage_area)
    
    # Ensure the result is a DataFrame
    specific_daily_flow[basin] = pd.DataFrame(log_transformed_values, columns= df.columns, index=df.index)
    
    # Scale the DataFrame using the loaded scaler
    specific_daily_flow[basin] = pd.DataFrame(daily_flow_scaler.transform(specific_daily_flow[basin].values), columns=df.columns, index=df.index)
    
climatological_flows = pd.read_pickle('/data/Hydra_Work/Sep_2024_Data/Climatological_flow.pkl')

# does the same type of normalisation as the daily flow
specific_climatological_flows = climatological_flows.copy()
for basin, df in specific_climatological_flows.items():
    drainage_area = static_variables.loc[basin, 'drainage_area']
    # Compute the log-transformed values
    log_transformed_values = np.log(df/ drainage_area)
    
    # Ensure the result is a DataFrame
    specific_climatological_flows[basin] = pd.DataFrame(log_transformed_values, columns=df.columns, index=df.index) 
    # Applies daily_flow scaler to each column of the DataFrame
    scaled_columns = {}
    for column in specific_climatological_flows[basin].columns:
        # Reshape to 2D array for scaler transformation
        column_values = specific_climatological_flows[basin][[column]].values
        scaled_column_values = daily_flow_scaler.transform(column_values)
        scaled_columns[column] = scaled_column_values.flatten()  # Flatten back to 1D array
    
    # Construct the scaled DataFrame
    specific_climatological_flows[basin] = pd.DataFrame(scaled_columns, index=df.index)


In [97]:
import joblib

# Define file paths

climatological_flows_filename = '/data/Hydra_Work/Sep_2024_Data/scaled_data_Sep_05/normalized_climatological_flows.pkl'
daily_flow_filename = '/data/Hydra_Work/Sep_2024_Data/scaled_data_Sep_05/normalized_daily_flow.pkl'


if not os.path.exists('/data/Hydra_Work/Sep_2024_Data/scaled_data_Sep_05/'):
    os.makedirs('/data/Hydra_Work/Sep_2024_Data/scaled_data_Sep_05/')
# Save the dataframes using joblib
joblib.dump(specific_climatological_flows, climatological_flows_filename)
joblib.dump(specific_daily_flow, daily_flow_filename)

['/data/Hydra_Work/Sep_2024_Data/scaled_data_Sep_05/normalized_daily_flow.pkl']

In [ ]:

scaler_directory = '/data/Hydra_Work/Sep_2024_Data/scalers'

# Doing log of specific discharge, normalised

era5 = pd.read_pickle('/data/Hydra_Work/Rodeo_Data/ERA5_Reanalysis_Pickles/All_Years_ERA5.pkl')

era5_scaler_filename = os.path.join(scaler_directory, 'era5_scaler.save')
era5_scaler = joblib.load(era5_scaler_filename)

# Remove Runoff from ERA5
for key, value in My_ERA5.items():
    era5[key].drop('ro', axis =1, inplace = True)
    
for basin, df in era5.items():
    # Scale the DataFrame using the loaded scaler
    era5[basin] = pd.DataFrame(era5_scaler.transform(df), columns=df.columns, index=df.index)
    
daily_flow_scaler_filename = os.path.join(scaler_directory, 'daily_flow_scaler.save')
daily_flow_scaler = joblib.load(daily_flow_scaler_filename)

daily_flow = pd.read_pickle('/data/Hydra_Work/Sep_2024_Data/USGS_flow.pkl') # Has Swwetwater but not Fonetenelle
specific_daily_flow = daily_flow.copy()
for basin, df in specific_daily_flow.items():
    drainage_area = static_variables.loc[basin, 'drainage_area']
    # Compute the log-transformed values
    log_transformed_values = np.log(df['00060_Mean'] / drainage_area)
    
    # Ensure the result is a DataFrame
    specific_daily_flow[basin] = pd.DataFrame(log_transformed_values, columns= df.columns, index=df.index)
    
    # Scale the DataFrame using the loaded scaler
    specific_daily_flow[basin] = pd.DataFrame(daily_flow_scaler.transform(specific_daily_flow[basin].values), columns=df.columns, index=df.index)
    
climatological_flows = pd.read_pickle('/data/Hydra_Work/Sep_2024_Data/Climatological_flow.pkl')

# does the same type of normalisation as the daily flow
specific_climatological_flows = climatological_flows.copy()
for basin, df in specific_climatological_flows.items():
    drainage_area = static_variables.loc[basin, 'drainage_area']
    # Compute the log-transformed values
    log_transformed_values = np.log(df/ drainage_area)
    
    # Ensure the result is a DataFrame
    specific_climatological_flows[basin] = pd.DataFrame(log_transformed_values, columns=df.columns, index=df.index) 
    # Applies daily_flow scaler to each column of the DataFrame
    scaled_columns = {}
    for column in specific_climatological_flows[basin].columns:
        # Reshape to 2D array for scaler transformation
        column_values = specific_climatological_flows[basin][[column]].values
        scaled_column_values = daily_flow_scaler.transform(column_values)
        scaled_columns[column] = scaled_column_values.flatten()  # Flatten back to 1D array
    
    # Construct the scaled DataFrame
    specific_climatological_flows[basin] = pd.DataFrame(scaled_columns, index=df.index)

In [ ]:
static_variables_filename = '/data/Hydra_Work/Sep_2024_Data/static_variables_original.csv'
static_variables.to_csv(static_variables_filename, index=True)

from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()
normalized_data = scaler.fit_transform(static_variables)
# Convert the normalized data back to a DataFrame
normalized_static_variables = pd.DataFrame(normalized_data, columns=static_variables.columns, index=static_variables.index)
normalized_static_variables_filename = '/data/Hydra_Work/Sep_2024_Data/scaled_data/static_variables_normalized.csv'
normalized_static_variables.to_csv(normalized_static_variables_filename)


In [ ]:
import joblib

# Define file paths
era5_filename = '/data/Hydra_Work/Sep_2024_Data/scaled_data/normalized_era5.pkl'
climatological_flows_filename = '/data/Hydra_Work/Sep_2024_Data/scaled_data/normalized_climatological_flows.pkl'
daily_flow_filename = '/data/Hydra_Work/Sep_2024_Data/scaled_data/normalized_daily_flow.pkl'

# Save the dataframes using joblib
joblib.dump(era5, era5_filename)
joblib.dump(specific_climatological_flows, climatological_flows_filename)
joblib.dump(specific_daily_flow, daily_flow_filename)